# 03 Sales Forecasting and Prediction
Professional forecasting notebook using Linear Regression.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

orders=pd.read_csv('../Dataset/orders.csv')
orders['Date']=pd.to_datetime(orders['Date'])

monthly=orders.groupby(pd.Grouper(key='Date',freq='MS'))['Sales'].sum().reset_index()
monthly=monthly.sort_values('Date')
monthly['Month_Index']=range(1,len(monthly)+1)

print(monthly.head())


## Train/Test Split

In [ ]:

train=monthly.iloc[:-3]
test=monthly.iloc[-3:]

X_train=train[['Month_Index']]
y_train=train['Sales']

X_test=test[['Month_Index']]
y_test=test['Sales']


## Train Linear Regression

In [ ]:

model=LinearRegression()
model.fit(X_train,y_train)

pred=model.predict(X_test)

print("MAE :",round(mean_absolute_error(y_test,pred),2))
print("RMSE:",round(np.sqrt(mean_squared_error(y_test,pred)),2))
print("R2  :",round(r2_score(y_test,pred),3))


## Forecast Next 6 Months

In [ ]:

future_index=range(monthly['Month_Index'].max()+1,
                   monthly['Month_Index'].max()+7)

future=pd.DataFrame({'Month_Index':future_index})
future['Forecast_Sales']=model.predict(future)

last_date=monthly['Date'].max()
future['Date']=pd.date_range(last_date+pd.offsets.MonthBegin(),
                             periods=6,
                             freq='MS')

print(future)


## Visualization

In [ ]:

plt.figure(figsize=(10,5))
plt.plot(monthly['Date'],monthly['Sales'],marker='o',label='Actual')
plt.plot(test['Date'],pred,marker='s',label='Predicted Test')
plt.plot(future['Date'],future['Forecast_Sales'],marker='^',label='Future Forecast')
plt.title('Monthly Sales Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Business Recommendations

In [ ]:

trend='increasing' if model.coef_[0]>0 else 'decreasing'

print("Business Interpretation")
print("- Overall sales trend:",trend)
print("- Forecast generated for the next 6 months.")
print("- Use forecasts for inventory planning.")
print("- Review staffing and marketing before peak months.")
print("- Compare actual sales with forecast every month to improve planning.")


## Save Forecast

In [ ]:

future.to_csv('../Dataset/sales_forecast.csv',index=False)
print("Forecast saved successfully.")
